Setup and Load Data

In [2]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

# Locate project root
project_root = Path.cwd()

if project_root.name == "notebook":
    project_root = project_root.parent

# Connect to SQLite database
db_path = project_root / "data" / "instamart.db"
conn = sqlite3.connect(db_path)

# Load required tables
customers_df = pd.read_sql_query(
    "SELECT * FROM customers", conn
)

products_df = pd.read_sql_query(
    "SELECT * FROM products", conn
)

orders_df = pd.read_sql_query(
    "SELECT * FROM orders", conn
)

print("Data loaded successfully!")
print("Customers:", customers_df.shape)
print("Products:", products_df.shape)
print("Orders:", orders_df.shape)

Data loaded successfully!
Customers: (1200, 5)
Products: (40, 4)
Orders: (12309, 9)


City vs Product Category Revenue using pivot_table()

In [3]:
# Merge Orders with Customers and Products

eda_data = (
    orders_df
    .merge(customers_df, on="customer_id", how="left")
    .merge(products_df, on="product_id", how="left")
)

# Consider only completed orders
completed_data = eda_data[
    eda_data["status"] == "completed"
].copy()

# Create City vs Product Category Revenue Pivot Table
city_category_revenue = pd.pivot_table(
    completed_data,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

print("City-wise Product Category Revenue:")
display(city_category_revenue)

City-wise Product Category Revenue:


category,Dairy,Electronics,Grocery,Personal Care,Snacks
city,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10
Pune,90349.55,187629.25,108456.60,34275.15,80207.50


Validation

In [4]:
# Validate Pivot Table against SQL Query 2

pivot_totals = city_category_revenue.sum(axis=0).sort_index()

sql_category_totals = pd.read_sql_query(
    """
    SELECT
        p.category,
        ROUND(SUM(o.order_amount), 2) AS total_revenue
    FROM orders o
    JOIN products p
        ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY p.category
    """,
    conn
).set_index("category")["total_revenue"].sort_index()

comparison = pd.DataFrame({
    "Pivot Revenue": pivot_totals.round(2),
    "SQL Revenue": sql_category_totals.round(2)
})

comparison["Match"] = np.isclose(
    comparison["Pivot Revenue"],
    comparison["SQL Revenue"],
    atol=0.01
)

display(comparison)

print("All categories match:", comparison["Match"].all())

,Pivot Revenue,SQL Revenue,Match
category,,,
Dairy,531938.95,531938.95,True
Electronics,1219248.10,1219248.10,True
Grocery,614293.30,614293.30,True
Personal Care,168433.95,168433.95,True
Snacks,464754.50,464754.50,True


All categories match: True


City-wise Aggregation using groupby() and Named Aggregation

In [5]:
# City-wise GroupBy Analysis

city_analysis = (
    completed_data
    .groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        total_orders=("order_id", "count")
    )
    .reset_index()
)

city_analysis["total_revenue"] = (
    city_analysis["total_revenue"].round(2)
)

city_analysis["average_order_amount"] = (
    city_analysis["average_order_amount"].round(2)
)

print("City-wise Business Analysis:")
display(city_analysis)

City-wise Business Analysis:


,city,total_revenue,average_order_amount,total_orders
0,Bengaluru,523643.40,276.47,1894
1,Chennai,463629.15,272.72,1700
2,Delhi,502829.80,243.62,2064
3,Hyderabad,528589.90,284.95,1855
4,Mumbai,479058.50,297.00,1613
5,Pune,500918.05,252.48,1984


Mean, Median and Skewness

In [6]:
# Statistical Analysis of Completed Order Amounts

order_amount_stats = completed_data["order_amount"].agg(
    ["mean", "median", "skew"]
)

print("Order Amount Statistical Summary:")
display(order_amount_stats)

print("\nInterpretation:")
print("Mean:", round(completed_data["order_amount"].mean(), 2))
print("Median:", round(completed_data["order_amount"].median(), 2))
print("Skewness:", round(completed_data["order_amount"].skew(), 4))

Order Amount Statistical Summary:


mean      269.907183
median    135.000000
skew        4.608827
Name: order_amount, dtype: float64


Interpretation:
Mean: 269.91
Median: 135.0
Skewness: 4.6088


Q1, Q3, IQR and Outlier Fences

In [7]:
# Calculate Quartiles and IQR

Q1 = completed_data["order_amount"].quantile(0.25)
Q3 = completed_data["order_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - (1.5 * IQR)
upper_fence = Q3 + (1.5 * IQR)

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower Fence:", round(lower_fence, 2))
print("Upper Fence:", round(upper_fence, 2))

Q1: 64.8
Q3: 297.0
IQR: 232.2
Lower Fence: -283.5
Upper Fence: 645.3


Outlier Count

In [8]:
# Identify Outliers using IQR Method

outliers = completed_data[
    (completed_data["order_amount"] < lower_fence) |
    (completed_data["order_amount"] > upper_fence)
]

outlier_count = len(outliers)

total_orders = len(completed_data)

print("Total Completed Orders:", total_orders)
print("Outlier Count:", outlier_count)
print(
    "Outlier Percentage:",
    round((outlier_count / total_orders) * 100, 2),
    "%"
)

Total Completed Orders: 11110
Outlier Count: 897
Outlier Percentage: 8.07 %


Correlation Matrix

In [9]:
# Correlation Analysis

correlation_data = completed_data[
    [
        "order_amount",
        "discount_pct",
        "quantity",
        "delivery_minutes"
    ]
]

correlation_matrix = correlation_data.corr()

print("Correlation Matrix:")
display(correlation_matrix.round(3))

Correlation Matrix:


,order_amount,discount_pct,quantity,delivery_minutes
order_amount,1.000,-0.095,0.087,-0.003
discount_pct,-0.095,1.000,-0.000,-0.010
quantity,0.087,-0.000,1.000,-0.013
delivery_minutes,-0.003,-0.010,-0.013,1.000


In [10]:
city_category_revenue = pd.pivot_table(
    completed_data,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

display(city_category_revenue)

print(
    "Pandas Grand Total:",
    round(city_category_revenue.loc["All", "All"], 2)
)

category,Dairy,Electronics,Grocery,Personal Care,Snacks,All
city,,,,,,
Bengaluru,87075.05,233909.45,110915.65,20193.85,71549.40,523643.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23471.50,90879.45,502829.80
Hyderabad,83731.00,228913.10,109475.40,24800.05,81670.35,528589.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187629.25,108456.60,34275.15,80207.50,500918.05
All,531938.95,1219248.10,614293.30,168433.95,464754.50,2998668.80


Pandas Grand Total: 2998668.8


In [11]:
amount = completed_data["order_amount"]

print("Outliers above upper fence:", (amount > 645.3).sum())
print("Outliers below lower fence:", (amount < -283.5).sum())

Outliers above upper fence: 897
Outliers below lower fence: 0


## Exploratory Data Analysis – Key Findings

Exploratory Data Analysis (EDA) was performed on 11,110 completed orders to understand revenue distribution, customer purchasing patterns, and relationships between order variables.

The city-wise product category revenue pivot table was validated against SQL results. All 30 city-category combinations matched, and the Pandas grand total of ₹2,998,668.80 exactly matched the SQL revenue total. Hyderabad generated the highest city-wise revenue (₹528,589.90), while Mumbai recorded the highest average order amount (₹297.00). Delhi had the highest number of completed orders (2,064). Electronics was the highest revenue-generating category, contributing ₹1,219,248.10.

The average order amount was ₹269.91, whereas the median was ₹135.00. A skewness value of 4.6088 indicates a strongly right-skewed distribution, suggesting that a relatively small number of high-value orders increase the average.

Using the IQR method, the lower and upper fences were -₹283.50 and ₹645.30, respectively. A total of 897 orders (8.07%) were identified as potential outliers above the upper fence, while no outliers were found below the lower fence. These records were retained for further investigation.

The correlation analysis showed very weak linear relationships among order amount, discount percentage, quantity, and delivery time. Order amount had a correlation of -0.095 with discount percentage, +0.087 with quantity, and -0.003 with delivery time. These results indicate weak linear associations and do not establish causation.

Overall, the analysis provides useful insights into revenue patterns, customer purchasing behaviour, and order performance for subsequent predictive modelling and business decision-making.